# Alzheimer's MRI Classification - Training on Google Colab

This notebook allows you to train the improved Alzheimer's detection model using Google Colab's GPU. 

## Step 1: Connect to GPU
Go to **Edit** -> **Notebook settings** -> Select **GPU** under Hardware accelerator.

## Step 2: Upload Your Data
You have two options:
1. **Upload Zip**: Zip your `dataset` folder, name it `dataset.zip`, and upload it to the Colab file explorer on the left.
2. **Google Drive**: Upload the dataset to your Drive and mount it here.

In [ ]:
# Run this cell if you uploaded 'dataset.zip'
!unzip -q dataset.zip -d .
print("Dataset unzipped!")

## Step 3: Install Dependencies

In [ ]:
!pip install tensorflow scikit-learn matplotlib

## Step 4: Define and Run Training Logic
This cell contains the full logic from `train_improved.py`.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.utils.class_weight import compute_class_weight

tf.random.set_seed(42)
np.random.seed(42)

# -- Config --
DATASET_DIR      = "dataset/train/"  # Ensure this path matches your unzipped structure
MODEL_DIR        = "model"
BEST_MODEL_PATH  = os.path.join(MODEL_DIR, "best_model.h5")
FINAL_MODEL_PATH = os.path.join(MODEL_DIR, "model.h5")
IMG_SIZE         = (224, 224)
BATCH_SIZE       = 16
NUM_CLASSES      = 4
EPOCHS           = 30

os.makedirs(MODEL_DIR, exist_ok=True)

# -- Data generators --
print("Building data generators...")

def mobilenet_preprocess(x):
    return preprocess_input(x)

train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=mobilenet_preprocess,
    validation_split=0.2,
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    shear_range=0.1,
    fill_mode='nearest'
)

val_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=mobilenet_preprocess,
    validation_split=0.2
)

train_gen = train_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=42
)

val_gen = val_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=42
)

class_names = list(train_gen.class_indices.keys())
print(f"Classes     : {class_names}")
print(f"Train images: {train_gen.samples}")
print(f"Val   images: {val_gen.samples}")

# -- Class weights --
print("\nComputing class weights...")
y_train = train_gen.classes
weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(weights))

# -- Model --
print("\nBuilding MobileNetV2 model...")
base = MobileNetV2(include_top=False, weights='imagenet', input_shape=(*IMG_SIZE, 3))
base.trainable = True

inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))
x = base(inputs, training=True)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(512, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(5e-4))(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(5e-4))(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)
model.compile(
    optimizer=optimizers.Adam(learning_rate=5e-5),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

cb = [
    callbacks.ModelCheckpoint(BEST_MODEL_PATH, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1),
    callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=4, min_lr=1e-7, verbose=1)
]

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    class_weight=class_weight_dict,
    callbacks=cb,
    verbose=1
)

model.save(FINAL_MODEL_PATH)
print("Training complete and model saved!")

## Step 5: Visualize Results

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.title('Loss')
plt.legend()

plt.show()

## Step 6: Download the Trained Model
Run the following cell to download the `model.h5` file to your computer.

In [ ]:
from google.colab import files
files.download('model/model.h5')